# Project 3 Milestone 2: Orbit Input Preparation

This notebook prepares the candidate-level input table required for downstream orbital characterization.

The goal is not to integrate orbits yet. Instead, this milestone verifies whether the cross-method candidate sample contains the astrometric, radial-velocity, and Galactocentric velocity fields needed for later orbit calculations.

Project 3 Milestone 1 was a planning/setup milestone, so the main data input for this milestone comes from the Project 2 candidate-level outputs.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("..").resolve()
DATA_PROCESSED = ROOT / "data" / "processed"
REPORT_DIR = ROOT / "report"

print("ROOT:", ROOT)
print("DATA_PROCESSED exists:", DATA_PROCESSED.exists())
print("REPORT_DIR exists:", REPORT_DIR.exists())


In [ ]:
print("Available processed CSV files:")
for path in sorted(DATA_PROCESSED.glob("*.csv")):
    print("-", path.name)


In [ ]:
cross_method_path = DATA_PROCESSED / "project2_candidate_cross_method_summary.csv"

assert cross_method_path.exists(), f"Missing input file: {cross_method_path}"

cross_df = pd.read_csv(cross_method_path)

print("cross-method summary:", cross_df.shape)
display(cross_df.head())
print("\nColumns:")
print(list(cross_df.columns))


## Field availability check

The minimum orbit-input fields checked here are:

- sky position: `ra`, `dec`
- distance proxy: `parallax`
- proper motions: `pmra`, `pmdec`
- line-of-sight velocity: `radial_velocity`
- Galactocentric velocity components: `galcen_vx`, `galcen_vy`, `galcen_vz`

If some fields are unavailable, the milestone still records this explicitly in the readiness summary.


In [ ]:
target_orbit_fields = [
    "source_id",
    "ra",
    "dec",
    "parallax",
    "pmra",
    "pmdec",
    "radial_velocity",
    "galcen_vx",
    "galcen_vy",
    "galcen_vz",
    "galcen_vtot",
]

field_availability = []
for field in target_orbit_fields:
    field_availability.append({
        "field": field,
        "available": field in cross_df.columns,
        "n_missing": int(cross_df[field].isna().sum()) if field in cross_df.columns else None,
        "fraction_missing": float(cross_df[field].isna().mean()) if field in cross_df.columns else None,
    })

field_availability_df = pd.DataFrame(field_availability)
display(field_availability_df)


In [ ]:
merged = cross_df.copy()

astrometric_required = [
    "ra",
    "dec",
    "parallax",
    "pmra",
    "pmdec",
    "radial_velocity",
]

galcen_velocity_required = [
    "galcen_vx",
    "galcen_vy",
    "galcen_vz",
]

orbit_required = astrometric_required + galcen_velocity_required

for col in orbit_required + ["galcen_vtot"]:
    if col not in merged.columns:
        merged[col] = np.nan

merged["has_astrometric_6d_input"] = merged[astrometric_required].notna().all(axis=1)
merged["has_galcen_velocity_input"] = merged[galcen_velocity_required].notna().all(axis=1)
merged["orbit_input_ready"] = merged[orbit_required].notna().all(axis=1)

readiness_summary = {
    "n_candidates": len(merged),
    "n_has_astrometric_6d_input": int(merged["has_astrometric_6d_input"].sum()),
    "n_has_galcen_velocity_input": int(merged["has_galcen_velocity_input"].sum()),
    "n_orbit_input_ready": int(merged["orbit_input_ready"].sum()),
    "fraction_orbit_input_ready": float(merged["orbit_input_ready"].mean()) if len(merged) else np.nan,
}

readiness_summary


In [ ]:
missing_summary = (
    merged[orbit_required]
    .isna()
    .sum()
    .rename("n_missing")
    .reset_index()
    .rename(columns={"index": "field"})
)

missing_summary["fraction_missing"] = missing_summary["n_missing"] / len(merged)

display(missing_summary)
display(merged[["has_astrometric_6d_input", "has_galcen_velocity_input", "orbit_input_ready"]].value_counts().reset_index(name="n"))


In [ ]:
preferred_columns = [
    "source_id",
    "ra",
    "dec",
    "parallax",
    "pmra",
    "pmdec",
    "radial_velocity",
    "bp_rp",
    "absolute_g_mag",
    "feh",
    "tangential_velocity_kms",
    "galcen_vx",
    "galcen_vy",
    "galcen_vz",
    "galcen_vtot",
    "candidate_strength",
    "cross_method_evidence_level",
    "n_methods_detected",
    "detected_by_dbscan",
    "detected_by_umap_dbscan",
    "detected_by_chemo_kinematic",
    "has_astrometric_6d_input",
    "has_galcen_velocity_input",
    "orbit_input_ready",
]

available_columns = [col for col in preferred_columns if col in merged.columns]

orbit_input = merged[available_columns].copy()

orbit_input_path = DATA_PROCESSED / "project3_orbit_input_candidates.csv"
orbit_input.to_csv(orbit_input_path, index=False)

print("Wrote:", orbit_input_path)
print("Shape:", orbit_input.shape)
display(orbit_input.head())
display(orbit_input["orbit_input_ready"].value_counts(dropna=False))


In [ ]:
report_path = REPORT_DIR / "project3_milestone2_orbit_input_preparation.md"

summary_lines = [
    "# Project 3 Milestone 2: Orbit Input Preparation",
    "",
    "## Goal",
    "",
    "This milestone prepares the candidate-level input table required for downstream orbital characterization.",
    "It does not perform orbit integration yet. Instead, it verifies whether the candidate sample contains the astrometric, radial-velocity, and Galactocentric velocity fields needed for later orbit analysis.",
    "",
    "Project 3 Milestone 1 was a planning/setup milestone. Therefore, this milestone uses the Project 2 cross-method candidate summary as the main candidate-level data input.",
    "",
    "## Input table",
    "",
    f"- Project 2 cross-method candidate summary: `{cross_method_path.name}`",
    "",
    "## Output table",
    "",
    f"- `{orbit_input_path.name}`",
    "",
    "## Orbit-readiness summary",
    "",
    f"- Number of candidates: {readiness_summary['n_candidates']}",
    f"- Candidates with 6D astrometric input: {readiness_summary['n_has_astrometric_6d_input']}",
    f"- Candidates with Galactocentric velocity input: {readiness_summary['n_has_galcen_velocity_input']}",
    f"- Candidates ready for orbit input: {readiness_summary['n_orbit_input_ready']}",
    f"- Orbit-input readiness fraction: {readiness_summary['fraction_orbit_input_ready']:.3f}",
    "",
    "## Required fields checked",
    "",
]

for field in orbit_required:
    n_missing = int(missing_summary.loc[missing_summary["field"] == field, "n_missing"].iloc[0])
    frac_missing = float(missing_summary.loc[missing_summary["field"] == field, "fraction_missing"].iloc[0])
    summary_lines.append(f"- `{field}`: missing {n_missing} / {len(merged)} ({frac_missing:.3f})")

summary_lines += [
    "",
    "## Interpretation",
    "",
    "The resulting orbit-input table provides a controlled bridge between the cross-method candidate selection stage and the later orbital characterization stage.",
    "Candidates marked as `orbit_input_ready = True` contain the minimum astrometric, radial-velocity, and Galactocentric velocity information required for the next milestone.",
    "",
    "If some candidates are not orbit-input ready, they remain useful for photometric, chemical, or kinematic context, but should not be used for full orbit integration without additional data completion or quality checks.",
    "",
    "## Next step",
    "",
    "Project 3 Milestone 3 will use this prepared table to compute orbital diagnostics, such as angular-momentum-related quantities, orbital energy proxies, or full orbit integrations depending on package availability and data quality.",
    "",
]

report_path.write_text("\n".join(summary_lines), encoding="utf-8")

print("Wrote:", report_path)
